# Multi-energy phase retrieval
Minimal use of `phase_retrieval_core_multienergy.py`.

In [ ]:
import numpy as np
from library import phase_retrieval_core_multienergy as pr

In [ ]:
holograms = np.load("data/energy_holograms.npy")  # (n_energy, nx, ny)
mask_pixel = np.load("data/mask_pixel.npy")
supportmask = np.load("data/supportmask.npy")
energies_eV = np.load("data/energies_eV.npy")

In [ ]:
recipe = {
    # Per-energy update schedule repeated during every outer iteration.
    "inner_mode": ["HAPRE", "ER"],       # Algorithm stages; a string is accepted for one stage.
    "inner_Nit": [700, 50],               # Iterations in each inner stage.
    "outer_iterations": 100,              # Number of update-plus-projection cycles.
    "warmup_mode": ["HAPRE", "ER"],      # Independent stages before energies are coupled.
    "warmup_Nit": [700, 50],                      # Warmup iterations; zero disables warmup.
    "shuffle_energies": True,             # Randomize energy update order each outer cycle.
    "random_seed": None,                  # Seed for energy-order randomization.
    # Scalars are broadcast to all inner stages; lists customize each stage.
    "beta_zero": 0.5,                     # Beta-schedule starting/reference value.
    "beta_mode": "arctan",               # Beta schedule name or one array per stage.
    "alpha_zero": 0.0,                    # TV-step strength; zero disables TV.
    "alpha_mode": "const",               # Alpha schedule name or array.
    "TV_freq": 1e9,                       # TV update interval.
    "warmup_beta_zero": None,             # None inherits beta_zero.
    "warmup_beta_mode": None,             # None inherits beta_mode.
    "warmup_alpha_zero": None,            # None inherits alpha_zero.
    "warmup_alpha_mode": None,            # None inherits alpha_mode.
    "warmup_TV_freq": None,               # None inherits TV_freq.
    "plot_every": 1e9,                    # Error sampling/plot interval.
    "average_img": 1,                     # Number of best late iterates to average.
    "Fourier_last": True,                 # Finish each stage with its Fourier constraint.
    "final_fourier_constraint": True,     # Finish final outputs on measured amplitudes.
    "hologram_intensity_cutoff_vmin": -1, # Percentile baseline subtraction; negative disables it.
    # Cross-energy object projection.
    "projection_model": "svd",           # "none", "svd", or "rank1_spectral".
    "rank": 1,                            # Retained residual rank for the SVD model.
    "projection_every": 1,                # Outer-cycle interval between projections.
    "projection_relaxation": 1.0,         # 0 keeps current objects; 1 applies full projection.
    "projection_start": 0,                # First outer cycle eligible for projection.
    "projection_static_mode": "mean",    # Static object: "mean", "first", or "none".
    "energy_weights": None,               # Positive weight per energy; None means equal weights.
    "log_floor": 1e-12,                   # Object-magnitude floor before taking complex log.
    # Rank-one spectral constraints; used only by rank1_spectral.
    "spectral_constraint": "free",       # "free", "kk", "known_beta", or "known_beta_kk".
    "energy_values": energies_eV,         # Strictly increasing energy axis for KK operations.
    "known_beta_spectrum": None,          # Known absorption-like response spectrum.
    "known_delta_spectrum": None,         # Optional known dispersion-like response spectrum.
    "absorption_part": "real",           # Whether absorption occupies "real" or "imag" part.
    "kk_sign": 1.0,                       # Sign convention multiplying the KK result.
    "kk_subtract_baseline": True,         # Remove endpoint baseline before KK.
    "kk_normalize_input": False,          # Normalize absorption before KK.
    "known_beta_normalization": "none",  # "none", "maxabs", "l2", or "std".
    "fit_known_beta_scale": True,         # Fit known-spectrum multiplicative scale.
    "fit_known_beta_offset": True,        # Fit known-spectrum additive offset.
}

fields, components, bsmasks, errors = (
    pr.multi_energy_phase_retrieval_algorithm(
        holograms,
        mask_pixel,
        supportmask,
        multi_energy_recipe=recipe,
    )
)